# 单机 1/4/8×RTX 4090：DDP/FSDP Benchmark

这个 Notebook 只保留服务器实验的高层操作：配置、预检、运行和展示。命令构造、断点复用、超时、日志、重复聚合、质量门禁、CSV/JSON/PNG 落盘都由 `distributed_bench_utils.py` 和 `run_dist_bench.py` 完成。

可以顺序执行全部单元。已完成且 identity 匹配的 raw case 会自动复用；只有设置 `rerun_existing=True` 才会强制重跑。

## 0. 会话配置

8 卡服务器保持默认 `(1, 4, 8)`；4 卡服务器改成 `(1, 4)`。Notebook kernel 必须使用安装了当前项目、PyTorch CUDA、pandas 和 matplotlib 的同一个 Python 环境。

In [ ]:
import sys
from importlib import import_module
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'tests' / 'distributed_bench_utils.py').is_file()
)
sys.path.insert(0, str(ROOT / 'tests'))

bench_utils = import_module('distributed_bench_utils')
BenchmarkSettings = bench_utils.BenchmarkSettings
DistributedBenchmark = bench_utils.DistributedBenchmark

settings = BenchmarkSettings(
    world_sizes=(1, 4, 8),
    strategies=('ddp', 'fsdp'),
    model_config='configs/model_125m_moe.yaml',
    local_batch=4,
    capacity_batches=(1, 2, 4, 8, 16, 32),
    warmup_steps=10,
    measure_steps=30,
    repeats=3,
)
bench = DistributedBenchmark(
    root=ROOT,
    output='artifacts/distributed_benchmark/rtx4090_125m_moe',
    settings=settings,
)

## 1. 服务器预检与硬件清单

预检会验证当前 kernel 的 Python/CUDA、可见 GPU 数、`torch.distributed`、`nvidia-smi`、模型配置和全部拓扑 YAML。硬件清单另外保存 GPU、驱动、PCIe/NVLink 拓扑、Git commit 与 dirty 状态。

In [ ]:
preflight = bench.preflight()
inventory = bench.inventory()
preflight, inventory

## 2. Weak scaling

固定每卡 local batch，比较 1→4→8 卡的吞吐、P50/P95 step 延迟、扩展效率、数据等待和显存。默认同时运行 DDP 与 FSDP。每个 case 独立 `torchrun`，10 次 warmup、30 次测量、3 次重复。

In [ ]:
weak = bench.run_weak()
weak.show()

## 3. Capacity sweep

逐个扫描 local batch。每个 case 位于独立子进程，OOM、超时和其他失败会写入 failure 表而不会污染后续 case；图表给出各策略/卡数的最大成功 local batch 和成功 case 的显存曲线。

In [ ]:
capacity = bench.run_capacity()
capacity.show()

## 4. 只重新打开已有结果

重启 Notebook 后不需要重跑实验。使用下面两行重新加载 summary、表格和图片。若确实要无条件重跑 raw case，调用 `bench.run_weak(rerun_existing=True)` 或 `bench.run_capacity(rerun_existing=True)`。

In [ ]:
# bench.load('weak').show()
# bench.load('capacity').show()